# v3 DF26 -- Monte-Carlo recovery (Figs V1/V2), Colab / M1 runner

A thin demo of the full DF26 pipeline (Sec 12.2 oracle test): train the Block-1 value/policy networks, collect a (parameter, moments) dataset via the batched (GPU-default) Block-2 path, train the moment surrogates, and recover parameters with Levenberg-Marquardt. It then plots **Fig V1** (true vs fitted moments) and **Fig V2** (true vs fitted parameters).

**One knob: the scale `PROFILE`.** Same code at every scale (see `src/v3/profiles.py`).

| profile | grids | sim | collect | use |
|---|---|---|---|---|
| `SMOKE` | 5x5x7 / 9x9x7 | 300x50 | 32 | tests / seconds |
| `MEDIUM` | 7x10x15 / 25x25x15 | 2000x120 | 400 | CPU / M1, minutes |
| `FULL` | 11x15x35 / 81x91x71 | 5000x300 | 10000 | paper grade, a CUDA GPU |

**Device.** `device="auto"` runs the float64 numerics on the GPU under CUDA (Colab: Runtime -> Change runtime type -> GPU) and on the CPU on Apple Metal / no-GPU (float64 has no reliable Metal kernel). `FULL` is intended for a CUDA GPU; on M1 it is an overnight/rented-GPU run.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # TFP needs Keras 2 semantics; pin before importing TF

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Pinned stack matching the validated local environment.
    !pip -q install "tensorflow==2.16.2" "tensorflow-probability==0.24.0" "tf-keras==2.16.0"
    # Make the repo importable: clone it (set your URL) or mount Drive and point REPO_ROOT at it.
    REPO_ROOT = "/content/DL_corp_finance"
    if not os.path.exists(REPO_ROOT):
        # !git clone <YOUR_REPO_URL> {REPO_ROOT}
        raise SystemExit("Set REPO_ROOT: git clone the repo or mount Drive, then rerun.")
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

import sys
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT, "| Colab:", IN_COLAB)

In [ ]:
import numpy as np
from src.v3.run import train_and_recover
from src.v3.validation import figures

PROFILE = "MEDIUM"   # "SMOKE" | "MEDIUM" | "FULL"
out = train_and_recover(profile=PROFILE, device="auto", verbose=True)

In [ ]:
print(f"device={out['device']} | profile={out['profile']} | kept draws={out['true_beta'].shape[0]}")
print("surrogate OOS R^2 (mean):", round(float(np.mean(out["surrogate_oos_r2"])), 3))
print("moment   R^2 (mean):", round(float(np.mean(out["moment_r2"])), 3), "  (Fig V1 gate: >= 0.99 at FULL)")
print("param    R^2:", dict(zip(figures.PARAM_NAMES, np.round(out["param_r2"], 3))))
print("           (Fig V2 gate: >= 0.95 for >= 7/8 params; chi is the weak one)")

In [ ]:
figures.plot_recovery_moments(out)   # Fig V1: true vs fitted moments
figures.plot_recovery_params(out)    # Fig V2: true vs fitted parameters

## Notes

- The Fig V1 (moment R^2 >= 0.99) and Fig V2 (param R^2 >= 0.95 for >= 7/8) acceptance gates are
  expected at `FULL` scale on a GPU; `MEDIUM` lands in the right region but below the paper gates.
  Only the components and the `SMOKE` end-to-end run are validated so far, so a `FULL` run is itself
  the gate verification: confirm at `MEDIUM`, then a reduced `FULL` canary, before an overnight run.

### Which Colab GPU for `FULL`

- **Pick an A100 (Colab Pro+).** This workload is float64-heavy: the dense policy-evaluation solve,
  Levenberg-Marquardt, and the weighting matrix all run in float64. Only datacenter GPUs (A100, V100)
  have full-rate float64; the T4 and L4 cripple float64 (1:32 to 1:64), so they run but the dense
  solves are an order of magnitude slower. The A100's 40 GB also fits the batched `[B, 5775, 5775]`
  solve and the `[B, 5000, 300]` panel.
- **Memory / batch.** `FULL` defaults to `collect_batch_size=8` (safe on 40 GB). On an A100 you can
  push toward 16 for throughput: `train_and_recover(profile="FULL")` after
  `from src.v3.profiles import get_profile` and editing, or pass a tuned profile. Do not raise it on a
  T4/L4.
- **Time.** Expect a multi-hour run (the 10k-row collection dominates). De-risk with a reduced FULL
  first: `train_and_recover("FULL", recovery_draws=4)` to confirm the R^2 trend, then scale up.

- Adaptive controller (Sec 6) demo: `from src.v3.run import run_adaptive_controller`.
- Reproducibility: every run is keyed by `master_seed` (Sec 10).
